<a href="https://www.kaggle.com/code/aleksandrmorozov123/ai-agents?scriptVersionId=246437259" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Implementation of reflection patterns on Groq models**

In [2]:
! pip install groq 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 3.4 MB/s eta 0:00:00


In [3]:
!pip install python-dotenv

In [4]:
import os
import re
from pprint import pprint
from groq import Groq
from dotenv import load_dotenv
from IPython.display import display_markdown
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Groq")

# Remember to load the environment variables. You should have the Groq API Key in there :)
load_dotenv()

client = Groq(api_key=secret_value_0)

In [5]:
generation_chat_history = [
    {
        "role": "system",
        "content": "You are a Python programmer tasked with generating high quality Python code."
        "Your task is to Generate the best content possible for the user's request. If the user provides critique," 
        "respond with a revised version of your previous attempt."
    }
]

In [6]:
generation_chat_history.append(
    {
        "role": "user",
        "content": "Generate a Python implementation of the Bucket Sort algorithm"
    }
)

In [7]:
bucketsort_code = client.chat.completions.create(
    messages=generation_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

generation_chat_history.append(
    {
        "role": "assistant",
        "content": bucketsort_code
    }
)

In [8]:
display_markdown(bucketsort_code, raw=True)

Here's a Python implementation of the Bucket Sort algorithm:

```Python
def bucket_sort(arr):
    # Create empty buckets
    buckets = []
    
    # Create empty lists and append them to the buckets
    for i in range(len(arr)):
        buckets.append([])
        
    # Insert elements into their respective buckets
    for j in arr:
        index = int(len(buckets) * j)
        buckets[int(index)].append(j)
        
    # Sort individual buckets using insertion sort
    for i in range(len(buckets)):
            insertion_sort(buckets[i])
            
    # Concatenate the result
    k = 0
    for i in range(len(buckets)):
        for j in range(len(buckets[i])):
            arr[k] = buckets[i][j]
            k += 1
    return arr

def insertion_sort(arr):
    for i in range(1, len(arr)):
        key = arr[i]
        j = i-1
        while j >= 0 and key < arr[j]:
                arr[j + 1] = arr[j]
                j -= 1
        arr[j + 1] = key
```
This code defines two functions: `bucket_sort` and `insertion_sort`. The `bucket_sort` function implements the bucket sort algorithm, and the `insertion_sort` function is used to sort individual buckets.

In [9]:
# reflection part
reflection_chat_history = [
    {
    "role": "system",
    "content": "You are an experienced phD computer scientist. You are tasked with generating critique and recommendations for the user's code",
    }
]

In [10]:
reflection_chat_history.append(
    {
        "role": "user",
        "content": bucketsort_code
    }
)

In [11]:
# generate critique to the python code
critique = client.chat.completions.create(
    messages=reflection_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

In [12]:
display_markdown(critique, raw=True)

Here's a thorough critique and some recommendations for improvement:

**Critique**

1. **bucket creation**: Instead of creating an empty list and appending empty lists to it, consider using a list comprehension to create the buckets directly.

2. **Index calculation**: The current method of calculating the bucket index can lead to `IndexError` if the input array contains values outside the range [0, 1). Consider using a more robust method to map the input range to the bucket indices.

3. **Bucket size**: The size of the buckets is equal to the length of the input array. However, this is not always the most efficient choice. Consider using a smaller number of buckets, and dynamically adjust the bucket size based on the input data.

5. **Insertion sort is not the most efficient choice for sorting individual buckets, especially for larger buckets. Consider using a more efficient sorting algorithm like quicksort or mergesort.

7. **Final concatenation**: The code does not handle the case where the input array is already sorted. Adding a check at the beginning of the function could avoid unnecessary computations.

**Recommendations**

Here's an improved version of the code:
```python
def bucket_sort(arr):
    if not arr or len(arr) <= 1:  # Early exit for empty or single-element arrays
        return arr
    
    # Calculate the number of buckets based on the input size
    num_buckets = int(len(arr) ** 0.5)
    
    # Create buckets with a more efficient method
    buckets = [[] for _ in range(num_buckets)]
    
    # Insert elements into their respective buckets with a more robust index calculation
    for val in arr:
        index = int(num_buckets * (val - min(arr)) / (max(arr) - min(arr)))
        buckets[index].append(val)
    
    # Sort individual buckets using a more efficient sorting algorithm
    for i in range(len(buckets)):
        buckets[i] = quicksort(buckets[i])
    
    # Concatenate the result
    return [val for bucket in buckets for val in bucket]

def quicksort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[0]
    less = [x for x in arr[1:] if x <= pivot]
    greater = [x for x in arr[1:] if x > pivot]
    return quicksort(less) + [pivot] + quicksort(greater)
```
**Changes**

* Improved bucket creation using a list comprehension
* Robust index calculation using the range of the input values)
* Dynamic bucket size calculation
* Early exit for empty or single-element arrays
* More efficient sorting algorithm for individual buckets (quicksort)
* Improved final concatenation using a list comprehension

In [13]:
# add critique to the generation chat history
generation_chat_history.append(
    {
        "role": "user",
        "content": critique
    }
)

In [14]:
# generation step 2
essay = client.chat.completions.create(
    messages=generation_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

In [15]:
display_markdown(essay, raw=True)

I appreciate your thorough critique and recommendations! Your suggested improvements make the code more efficient, and scalable.

Here's the revised code based on your recommendations:

```python
def bucket_sort(arr):
    if not arr or len(arr) <= 1:  
        return arr
    
    num_buckets = int(len(arr) ** 0.5)
    buckets = [[] for _ in range(num_buckets)]
    
    for val in arr:
        index = int(num_buckets * (val - min(arr)) / (max(arr) - min(arr)))
        buckets[index].append(val)
    
    for i in range(len(buckets)):
        buckets[i] = quicksort(buckets[i])
    
    return [val for bucket in buckets for val in bucket]

def quicksort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[0]
    less = [x for x in arr[1:] if x <= pivot]
    greater = [x for x in arr[1:] if x > pivot]
    return quicksort(less) + [pivot] + quicksort(greater)
```

This revised code incorporates all the recommended improvements, including:

*   Early exit for empty or single-element arrays
*   Dynamic bucket size calculation
*   Robust index calculation using the range of the input values
*   More efficient sorting algorithm for individual buckets (quicksort)
*   Improved final concatenation using a list comprehension

These changes enhance the code's performance, scalability, and maintainability. Thank you for your valuable feedback!

**Implement tool function for agent**

In [16]:
import json

def get_current_weather(location: str, unit: str):
	"""
	Get the current weather in a given location

	location (str): The city and state, e.g. Miami, Florida
	unit (str): The unit. It can take two values; "celsius", "fahrenheit"
	"""
	if location == "Miami":
		return json.dumps({"Fahrenheit": 75, "unit": unit})

	else:
		return json.dumps({"temperature": 130, "unit": unit})

In [17]:
get_current_weather(location="Miami", unit="Fahrenheit")

'{"Fahrenheit": 75, "unit": "Fahrenheit"}'

In [18]:
MODEL = "llama-3.3-70b-versatile"
GROQ_CLIENT = Groq(api_key=secret_value_0)

# Define the System Prompt as a constant
TOOL_SYSTEM_PROMPT = """
You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags. 
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug 
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.
For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>}
</tool_call>

Here are the available tools:

<tools> {
    "name": "get_current_weather",
    "description": "Get the current weather in a given location location (str): The city and state, e.g. Madrid, Barcelona unit (str): The unit. It can take two values; 'celsius', 'fahrenheit'",
    "parameters": {
        "properties": {
            "location": {
                "type": "str"
            },
            "unit": {
                "type": "str"
            }
        }
    }
}
</tools>
"""

In [19]:
# ask the question
tool_chat_history = [
    {
        "role": "system",
        "content": TOOL_SYSTEM_PROMPT
    }
]
agent_chat_history = []

user_msg = {
    "role": "user",
    "content": "What's the current temperature in Miami, in Fahrenheit?"
}

tool_chat_history.append(user_msg)
agent_chat_history.append(user_msg)

output = GROQ_CLIENT.chat.completions.create(
    messages=tool_chat_history,
    model=MODEL
).choices[0].message.content

print(output)

<tool_call>
{"name": "get_current_weather","arguments": {"location": "Miami", "unit": "fahrenheit"}}
</tool_call>


In [20]:
def parse_tool_call_str(tool_call_str: str):
    pattern = r'</?tool_call>'
    clean_tags = re.sub(pattern, '', tool_call_str)
    
    try:
        tool_call_json = json.loads(clean_tags)
        return tool_call_json
    except json.JSONDecodeError:
        return clean_tags
    except Exception as e:
        print(f"Unexpected error: {e}")
        return "There was some error parsing the Tool's output"

In [21]:
parsed_output = parse_tool_call_str(output)
parsed_output

{'name': 'get_current_weather',
 'arguments': {'location': 'Miami', 'unit': 'fahrenheit'}}

In [22]:
result = get_current_weather(**parsed_output["arguments"])

In [23]:
result

'{"Fahrenheit": 75, "unit": "fahrenheit"}'

In [24]:
agent_chat_history.append({
    "role": "user",
    "content": f"Observation: {result}"
})

In [25]:
GROQ_CLIENT.chat.completions.create(
    messages=agent_chat_history,
    model=MODEL
).choices[0].message.content

'The current temperature in Miami is 75°F.'